In [1]:
%reload_ext autoreload
%autoreload 2
import os, sys
import torch
import numpy as np
import pandas as pd

import torch.utils.data as data_utils
from functools import partial
from torch.utils.data import DataLoader

from searchspace import NDS
import pycls.datasets.loader as loader

# search space available on https://dl.fbaipublicfiles.com/nds/data.zip
# https://github.com/facebookresearch/nds?tab=readme-ov-file


In [2]:
# setup & hyperparameters
_selected_ss = 3
RANDOM_SEED = 4 

# search space sepectific
search_spaces = ['DARTS', 'ENAS', 'PNAS', 'NASNet', 'Amoeba']
search_space_str = search_spaces[_selected_ss]
# batch size / dataset shape
BATCH_SIZE = 64
_input_shape = (BATCH_SIZE, 3, 16, 16)
coeff = ["Spearman's", "Kendall's"]

# seed + generator
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
rng_generator = np.random.default_rng(seed=RANDOM_SEED)

# set up device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# TODO change the path where you unpacked your zip file (Download at https://dl.fbaipublicfiles.com/nds/data.zip)
path = "./"
# path = "./data"
nds_searchpace_c = NDS(search_spaces[_selected_ss], path=path)
size_nds_ss = nds_searchpace_c.__len__()
_n_models = 500

# csv export
FOLDER = "../experiments/NDS_{}_cifar10_ch/".format(search_space_str)
os.makedirs(f"{FOLDER}",exist_ok=True)

In [5]:
# load models and acc
_models, accuracy = [], []
instances = rng_generator.choice(size_nds_ss, size=(_n_models), replace=False)
model = nds_searchpace_c.get_network(instances[10]).to(device)
acc = nds_searchpace_c.get_final_accuracy(instances[10])
print(f"Model 10: {model}, acc: {acc}")
#print(f"Accuracy range: {nds_searchpace_c.get_final_accuracy_range()}")
# for i in range(0, _n_models):
#     model = nds_searchpace_c.get_network(i).to(device)
#     acc = nds_searchpace_c.get_final_accuracy(i)
#     _models.append(model)
#     accuracy.append(acc)


Model 10: NetworkCIFAR(
  (stem): Sequential(
    (0): Conv2d(3, 72, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(72, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (cells): ModuleList(
    (0-5): 6 x Cell(
      (preprocess0): ReLUConvBN(
        (op): Sequential(
          (0): ReLU()
          (1): Conv2d(72, 24, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (preprocess1): ReLUConvBN(
        (op): Sequential(
          (0): ReLU()
          (1): Conv2d(72, 24, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (_ops): ModuleList(
        (0): Sequential(
          (0): ReLU()
          (1): Conv2d(24, 24, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): BatchNorm2d(24, eps=1e-0